In [1]:
from mibici.modeling.p1 import build_model_pipeline

model = build_model_pipeline()
display(model)

2026-10-18 13:20:42.193 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/mibici/mibici_predictions


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [2]:
from mibici.dataset import get_split_periods
from mibici.pipeline import load_stage

periods = get_split_periods()
display(periods)

data = load_stage("common", split="train")
trips = data["viajes"]

{'train': (Timestamp('2024-10-01 00:00:00'), Timestamp('2026-02-23 00:00:00')),
 'validation': (Timestamp('2026-02-23 00:00:00'),
  Timestamp('2026-06-12 00:00:00')),
 'test': (Timestamp('2026-06-12 00:00:00'), Timestamp('2026-10-01 00:00:00'))}

2026-10-18 13:20:57.464 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 13:20:57.464 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


In [3]:
from mibici.modeling.p1 import build_training_daily_data

train_start, train_end = periods["train"]

train_daily = build_training_daily_data(
    trips,
    start=train_start,
    end_exclusive=train_end,
)

display(train_daily.head())

,station_id,date,departures
0,2,2024-10-01,21
1,2,2024-10-02,74
2,2,2024-10-03,83
3,2,2024-10-04,63
4,2,2024-10-05,44


In [4]:
from mibici.features import build_p1_features

train_features = build_p1_features(train_daily)
display(train_features.head(10))

,station_id,date,departures,day_of_week,departures_lag_1,departures_lag_7,departures_mean_7
0,2,2024-10-01,21,1,NaN,NaN,NaN
1,2,2024-10-02,74,2,21.0,NaN,NaN
2,2,2024-10-03,83,3,74.0,NaN,NaN
3,2,2024-10-04,63,4,83.0,NaN,NaN
4,2,2024-10-05,44,5,63.0,NaN,NaN
5,2,2024-10-06,30,6,44.0,NaN,NaN
6,2,2024-10-07,76,0,30.0,NaN,NaN
7,2,2024-10-08,83,1,76.0,21.0,55.857143
8,2,2024-10-09,86,2,83.0,74.0,64.714286
9,2,2024-10-10,75,3,86.0,83.0,66.428571


In [5]:
from mibici.modeling.p1 import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
)

train_ready = train_features.dropna(
    subset=NUMERIC_FEATURES
).copy()

X_train = train_ready[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y_train = train_ready[TARGET]

print(f"Filas excluidas por falta de historia: " f"{len(train_features) - len(train_ready):,}")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

assert X_train.notna().all().all()
assert y_train.notna().all()

Filas excluidas por falta de historia: 2,597
X_train: (184944, 5)
y_train: (184944,)


In [6]:
validation_data = load_stage("common", split="validation")
validation_trips = validation_data["viajes"]

validation_start, validation_end = periods["validation"]

2026-10-18 13:21:07.407 | INFO     | mibici.dataset:load_raw:166 - Partición validation: 2026-02-23 a 2026-06-11
2026-10-18 13:21:07.407 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 1,231,240


In [7]:
import pandas as pd

known_stations = sorted(train_ready["station_id"].unique())

validation_calendar = pd.MultiIndex.from_product(
    [
        known_stations,
        pd.date_range(
            validation_start,
            validation_end - pd.Timedelta(days=1),
            freq="D",
        ),
    ],
    names=["station_id", "date"],
).to_frame(index=False)

validation_counts = (
    validation_trips.assign(
        date=validation_trips["Inicio_del_viaje"].dt.normalize()
    )
    .groupby(["Origen_Id", "date"])
    .size()
    .rename("departures")
    .reset_index()
    .rename(columns={"Origen_Id": "station_id"})
)

validation_daily = validation_calendar.merge(
    validation_counts,
    on=["station_id", "date"],
    how="left",
    validate="one_to_one",
)

validation_daily["departures"] = (
    validation_daily["departures"].fillna(0).astype("int64")
)

In [8]:
included_trips = validation_trips["Origen_Id"].isin(known_stations)

assert validation_daily["departures"].sum() == included_trips.sum()

print(f"Filas estación-día: {len(validation_daily):,}")
print(f"Viajes fuera del alcance: {(~included_trips).sum():,}")

display(validation_daily.head())

Filas estación-día: 40,439
Viajes fuera del alcance: 3,617


,station_id,date,departures
0,2,2026-02-23,11
1,2,2026-02-24,34
2,2,2026-02-25,54
3,2,2026-02-26,50
4,2,2026-02-27,55


In [9]:
history = train_daily.loc[
    train_daily["station_id"].isin(known_stations)
    & train_daily["date"].ge(
        validation_start - pd.Timedelta(days=7)
    )
    & train_daily["date"].lt(validation_start)
].copy()

validation_context = pd.concat(
    [history, validation_daily],
    ignore_index=True,
)

context_features = build_p1_features(validation_context)

validation_features = context_features.loc[
    context_features["date"].ge(validation_start)
    & context_features["date"].lt(validation_end)
].copy()

X_validation = validation_features[
    CATEGORICAL_FEATURES + NUMERIC_FEATURES
]
y_validation = validation_features[TARGET]

assert len(validation_features) == len(validation_daily)
assert X_validation.notna().all().all()
assert y_validation.notna().all()

print(f"X_validation: {X_validation.shape}")
print(f"y_validation: {y_validation.shape}")

X_validation: (40439, 5)
y_validation: (40439,)


In [10]:
from mibici.modeling.p1 import (
    build_model_pipeline,
    predict_departures,
)

model = build_model_pipeline()
model.fit(X_train, y_train)

validation_predictions = predict_departures(
    model,
    X_validation,
)

assert len(validation_predictions) == len(y_validation)
assert (validation_predictions >= 0).all()

print(f"Predicciones generadas: {len(validation_predictions):,}")

Predicciones generadas: 40,439


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_predictions = X_validation["departures_lag_7"]

results = pd.DataFrame([
    {
        "model": "Weekly baseline",
        "validation_mae": mean_absolute_error(
            y_validation, baseline_predictions
        ),
        "validation_rmse": mean_squared_error(
            y_validation, baseline_predictions
        ) ** 0.5,
    },
    {
        "model": "Linear regression",
        "validation_mae": mean_absolute_error(
            y_validation, validation_predictions
        ),
        "validation_rmse": mean_squared_error(
            y_validation, validation_predictions
        ) ** 0.5,
    },
])

display(results.round(3))

print(
    "Predicciones negativas:",
    (validation_predictions < 0).sum(),
)

,model,validation_mae,validation_rmse
0,Weekly baseline,8.468,16.846
1,Linear regression,7.110,13.180


Predicciones negativas: 0


In [12]:
results = pd.DataFrame([
    {
        "model": "Weekly baseline",
        "validation_mae": mean_absolute_error(
            y_validation, X_validation["departures_lag_7"]
        ),
        "validation_rmse": mean_squared_error(
            y_validation, X_validation["departures_lag_7"]
        ) ** 0.5,
    },
    {
        "model": "Linear regression — clipped",
        "validation_mae": mean_absolute_error(
            y_validation, validation_predictions
        ),
        "validation_rmse": mean_squared_error(
            y_validation, validation_predictions
        ) ** 0.5,
    },
])

display(results.round(3))

,model,validation_mae,validation_rmse
0,Weekly baseline,8.468,16.846
1,Linear regression — clipped,7.110,13.180


La regresión lineal supera la referencia semanal en validación.
Se adopta un límite inferior de cero para las predicciones,
con una mejora pequeña y resultados compatibles con el dominio.

In [13]:
from mibici.modeling.tracking import configure_tracking

experiment = configure_tracking()

print(f"Experimento: {experiment.name}")
print(f"ID: {experiment.experiment_id}")


/home/jovyan/notebooks/mibici/mibici_predictions/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/18 13:21:17 INFO mlflow.tracking.fluent: Experiment with name 'TC5061-Proyecto-Equipo40' does not exist. Creating a new experiment.


Experimento: TC5061-Proyecto-Equipo40
ID: 10


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

from mibici.modeling.tracking import log_evaluation

baseline_predictions = X_validation["departures_lag_7"]

baseline_run_id = log_evaluation(
    run_name="p1_weekly_baseline",
    params={
        "model": "weekly_baseline",
        "lag_days": 7,
        "horizon_days": 1,
        "evaluation_protocol": "daily_one_step",
        "validation_start": str(validation_start.date()),
        "validation_end_exclusive": str(validation_end.date()),
        "validation_rows": len(y_validation),
        "station_scope": "stations_with_training_history",
    },
    metrics={
        "validation_mae": mean_absolute_error(
            y_validation, baseline_predictions
        ),
        "validation_rmse": mean_squared_error(
            y_validation, baseline_predictions
        ) ** 0.5,
    },
    tags={
        "problem": "p1",
        "run_type": "baseline",
    },
)

print(f"Baseline registrado: {baseline_run_id}")

🏃 View run p1_weekly_baseline at: http://mlflow:5000/#/experiments/10/runs/1303e67353584129ad715af26ab14a94
🧪 View experiment at: http://mlflow:5000/#/experiments/10
Baseline registrado: 1303e67353584129ad715af26ab14a94
